# Check Market v2 (21-nov-2026)

Versión corregida del notebook «Check Market». Lo que sirve a diario ya está en el **Recession Monitor**, pestaña
[Big Tech y semis](https://boletix.github.io/recession-monitor/#tech): se calcula solo cada día con EDGAR, Yahoo y FRED.
Este cuaderno queda para explorar.

**Qué cambia frente al original**
- La clave de FRED ya no va escrita en el código: se lee de los *secrets* de Colab (`FRED_API_KEY`).
- `yahoo_fin`, Slickcharts y Finviz ya no responden (403): la lista y los pesos del S&P 500 salen de la cartera diaria del SPY.
- FAATMANN era una **suma de precios** con Alphabet dos veces; ahora es una cesta FAAMG por valor en bolsa.
- El peso histórico de FAATMANN se dividía por el total del **último día**; el TSR contaba los dividendos dos veces;
  las medias se calculaban después de recortar la serie; los porcentajes de amplitud dividían por valores sin datos;
  vacantes y paro se alineaban por posición y no por fecha. Detalle en cada celda.

In [ ]:
#@title Librerías, claves y utilidades
!pip -q install --upgrade yfinance fredapi openpyxl
import os, io, requests
import numpy as np, pandas as pd, matplotlib.pyplot as plt, yfinance as yf
from fredapi import Fred
try:
    from google.colab import userdata          # Colab → icono de la llave → FRED_API_KEY
    FRED_KEY = userdata.get('FRED_API_KEY')
except Exception:
    FRED_KEY = os.environ.get('FRED_API_KEY')
fred = Fred(api_key=FRED_KEY)
plt.rcParams.update({'figure.figsize': (16, 7), 'axes.grid': True, 'grid.alpha': .3})

def closes(tickers, start='2000-01-01', adjusted=False):
    """Cierres fecha × ticker. adjusted=True incluye dividendos (rentabilidad total).
    Close de Yahoo ya viene ajustado por splits."""
    d = yf.download(tickers, start=start, auto_adjust=False, progress=False, threads=True)
    c = d['Adj Close' if adjusted else 'Close']
    if isinstance(c, pd.Series):
        c = c.to_frame(tickers if isinstance(tickers, str) else tickers[0])
    if c.index.tz is not None:
        c.index = c.index.tz_localize(None)
    return c.dropna(how='all')

def sp500_members():
    """Valores y pesos (%) del S&P 500 hoy, de la cartera diaria del SPY (State Street)."""
    url = 'https://www.ssga.com/us/en/intermediary/library-content/products/fund-data/etfs/us/holdings-daily-us-en-spy.xlsx'
    raw = pd.read_excel(io.BytesIO(requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=60).content), header=None)
    h = raw.index[raw.iloc[:, 1].astype(str).str.strip() == 'Ticker'][0]
    df = raw.iloc[h + 1:].set_axis([str(c).strip() for c in raw.iloc[h]], axis=1)
    df = df[pd.to_numeric(df['Weight'], errors='coerce') > 0]
    df = df[df['Ticker'].astype(str).str.fullmatch(r'[A-Z]{1,5}(\.[A-Z])?')]
    return pd.Series(df['Weight'].astype(float).values, index=df['Ticker'].str.replace('.', '-', regex=False), name='peso')

## Monitor v1 (primera celda del original): retirado
Su score mezclaba el humor del mercado con la economía, ponía **0 (sin riesgo)** cuando faltaba un dato, llamaba «WMA 200»
a la media de 200 sesiones (el umbral del 12% era de la media de 200 semanas) y tenía el nivel 6.475 del S&P escrito a mano.
Lo sustituye el monitor v5: la economía en el score; VIX, petróleo, bono a 30 años y McClellan en las pestañas de mercado;
el crédito con el diferencial del high yield de FRED en lugar de LQD/HYG.

In [ ]:
#@title Curva de tipos: pendiente, curvatura y diferenciales
desde = '2005-01-01'  #@param {type:"date"}
y = closes(['^IRX', '^FVX', '^TNX', '^TYX'], start='2000-01-01')[['^IRX', '^FVX', '^TNX', '^TYX']].dropna()
mat = np.array([0.25, 5, 10, 30])
# Todas las regresiones de golpe (antes, un bucle por día que fallaba con cualquier hueco)
X = np.vstack([np.ones_like(mat), mat]).T
pend = pd.Series(np.linalg.lstsq(X, y.values.T, rcond=None)[0][1], index=y.index)
curv = pd.Series(np.polyfit(mat, y.values.T, 2)[0], index=y.index)
fig, ax = plt.subplots(3, 1, figsize=(16, 13), sharex=True)
pend.loc[desde:].plot(ax=ax[0], title='Pendiente (pp por año de plazo)'); ax[0].axhline(0, color='k', ls='--')
curv.loc[desde:].plot(ax=ax[1], title='Curvatura (término cuadrático)', color='tab:orange'); ax[1].axhline(0, color='k', ls='--')
for a, b, n in [('^TYX', '^TNX', '30A − 10A'), ('^TNX', '^IRX', '10A − 13 semanas'), ('^TNX', '^FVX', '10A − 5A')]:
    (y[a] - y[b]).loc[desde:].plot(ax=ax[2], label=n)
ax[2].axhline(0, color='k', ls='--'); ax[2].legend(); ax[2].set_title('Diferenciales (pp)')
plt.show()

In [ ]:
#@title TSR anual (con dividendos reinvertidos, en la divisa de cada acción)
# Antes: precios de history() ya ajustados por dividendos + la suma de los dividendos = contados dos veces.
def tsr(ticker, start='2010-01-01'):
    p = closes(ticker, start=start, adjusted=True).iloc[:, 0].dropna()
    años = (p.index[-1] - p.index[0]).days / 365.25
    return (p.iloc[-10:].mean() / p.iloc[:10].mean()) ** (1 / años) - 1

for t in ['SIKA.SW', 'KRX.IR', 'SGO.PA', 'FAST', 'ITW', 'CSL']:
    print(f'{t:8s} TSR {tsr(t):6.2%} al año desde 2010')

In [ ]:
#@title S&P 500 y Nasdaq frente a sus medias de 50, 100 y 200 sesiones
desde = '2019-06-01'  #@param {type:"date"}
ix = closes(['^GSPC', '^IXIC', '^VIX'], start='2010-01-01')
fig, ax = plt.subplots(2, 2, figsize=(26, 14))
for i, (t, n) in enumerate([('^GSPC', 'S&P 500'), ('^IXIC', 'Nasdaq')]):
    s = ix[t].dropna()
    ma = {k: s.rolling(k).mean() for k in (50, 100, 200)}       # con toda la serie y después se recorta
    s.loc[desde:].plot(ax=ax[i, 0], label=n, color='k')
    for k, m in ma.items():
        m.loc[desde:].plot(ax=ax[i, 0], label=f'MM{k}')
        (s / m - 1).loc[desde:].plot(ax=ax[i, 1], label=f'{n} vs MM{k}')
    ax[i, 0].set_title(f'{n} y sus medias'); ax[i, 0].legend()
    ax[i, 1].axhline(0, color='k', ls='--'); ax[i, 1].set_title(f'{n}: distancia a sus medias'); ax[i, 1].legend()
plt.show()

spx = ix['^GSPC'].dropna()
fig, ax = plt.subplots(2, 1, sharex=True)
ix['^VIX'].loc[desde:].plot(ax=ax[0], color='tab:blue', title='VIX'); ax[0].axhline(30, color='k', ls='--')
(spx / spx.cummax() - 1).loc[desde:].plot(ax=ax[1], color='salmon', title='S&P 500: caída desde su máximo histórico')
ax[1].axhline(-0.10, color='k', ls='--')
plt.show()

In [ ]:
#@title Equiponderado (RSP) frente al S&P 500: rentabilidad, beta y fuerza relativa
desde = '2021-04-18'  #@param {type:"date"}
ventana = 63  #@param {type:"number"}
px = closes(['RSP', 'SPY'], start='2010-01-01', adjusted=True).dropna()
r = px.pct_change().dropna()                     # las dos series alineadas por fecha
beta = (r['RSP'].rolling(ventana).cov(r['SPY']) / r['SPY'].rolling(ventana).var()).loc[desde:]
rel = (px['RSP'] / px['SPY']).loc[desde:]
fig, ax = plt.subplots(3, 1, figsize=(16, 14), sharex=True)
(px.loc[desde:] / px.loc[desde:].iloc[0] - 1).plot(ax=ax[0], title='Rentabilidad total desde la fecha')
beta.plot(ax=ax[1], title=f'Beta del equiponderado frente al S&P ({ventana} sesiones)'); ax[1].axhline(1, color='k', ls='--')
(rel / rel.iloc[0]).plot(ax=ax[2], color='tab:green', title='Equiponderado / S&P (sube = el mercado se ensancha)'); ax[2].axhline(1, color='k', ls='--')
plt.show()

In [ ]:
#@title FAAMG, semis, growth y value frente al S&P 500 · peso de FAAMG en el índice
desde = '2021-04-18'  #@param {type:"date"}
w = sp500_members()
wF = w.reindex(['META', 'AAPL', 'AMZN', 'MSFT', 'GOOGL']).fillna(0)
wF['GOOGL'] += w.get('GOOG', 0)                  # Alphabet una sola vez (antes GOOGL y GOOG)
p = closes(list(wF.index) + ['^GSPC', 'RSP', '^RLG', '^RLV', '^SOX'], start='2016-01-01').ffill()
# Cesta por valor en bolsa: «acciones» equivalentes = peso de hoy / precio de hoy (antes, suma de precios)
a = wF / p[wF.index].iloc[-1]
F = (p[wF.index] * a).sum(axis=1)
out = pd.DataFrame({'S&P 500': p['^GSPC'], 'Equiponderado': p['RSP'], 'Growth (R1000)': p['^RLG'],
                    'Value (R1000)': p['^RLV'], 'Semis (SOX)': p['^SOX'], 'FAAMG': F}).loc[desde:]
(out / out.iloc[0] - 1).plot(title='Rentabilidad en precio desde la fecha'); plt.axhline(0, color='k', lw=.8); plt.show()
# Peso aproximado: el de hoy, movido por el precio de la cesta frente al índice (sin cambios de acciones)
peso = wF.sum() * (F / F.iloc[-1]) / (p['^GSPC'] / p['^GSPC'].iloc[-1])
peso.plot(title=f'Peso aproximado de FAAMG en el S&P 500 (%) · hoy {wF.sum():.1f}%'); plt.show()
print(w.sort_values(ascending=False).head(15).round(2).to_string())

In [ ]:
#@title Vacantes y parados (JOLTS), alineados por fecha
d = pd.concat({'Vacantes': fred.get_series('JTSJOL', '2001-01-01'),
               'Parados': fred.get_series('UNEMPLOY', '2001-01-01')}, axis=1).dropna()
fig, ax = plt.subplots(2, 2, figsize=(24, 11))
d.plot(ax=ax[0, 0], title='Vacantes y parados (miles)')
d.loc['2020-08':].plot(ax=ax[0, 1], title='Desde agosto de 2020')
d['Vacantes'].diff().rolling(12).corr(d['Parados'].diff()).plot(ax=ax[1, 0], color='salmon',
    title='Correlación de 12 meses entre los cambios mensuales'); ax[1, 0].axhline(0, color='k', ls='--')
(d['Vacantes'] / d['Parados']).plot(ax=ax[1, 1], color='tab:green',
    title='Vacantes por parado (bajo 1: más parados que vacantes)'); ax[1, 1].axhline(1, color='k', ls='--')
plt.show()

In [ ]:
#@title Amplitud de los ~500 valores del S&P 500 (lista de hoy: sesgo de supervivencia)
inicio = '2015-01-01'  #@param {type:"date"}
desde = '2019-01-01'  #@param {type:"date"}
tickers = list(sp500_members().index)
px = closes(tickers, start=inicio).ffill(limit=3)
spx = closes('^GSPC', start=inicio).iloc[:, 0]
m50, m200 = px.rolling(50, min_periods=50).mean(), px.rolling(200, min_periods=200).mean()
hi, lo = px.rolling(252, min_periods=252).max(), px.rolling(252, min_periods=252).min()
def share(cond, valid):                          # % sobre los valores con datos ese día (antes, sobre todos)
    return (cond & valid).sum(axis=1) / valid.sum(axis=1) * 100
a200, a50 = share(px > m200, m200.notna()), share(px > m50, m50.notna())
nh, nl = share(px >= hi, hi.notna()), share(px <= lo, lo.notna())
both = share((px < m50) & (px < m200), m200.notna())
r = px.pct_change(fill_method=None)
up, down = (r > 0).sum(axis=1), (r < 0).sum(axis=1)  # un dato que falta ya no cuenta como bajada
rana = (up - down) / (up + down) * 1000
mcc = rana.ewm(span=19, adjust=False).mean() - rana.ewm(span=39, adjust=False).mean()
fig, ax = plt.subplots(5, 1, figsize=(18, 24), sharex=True)
a200.loc[desde:].plot(ax=ax[0], label='Sobre MM200'); a50.loc[desde:].plot(ax=ax[0], label='Sobre MM50', alpha=.6)
ax[0].axhline(50, color='k', ls='--'); ax[0].legend(); ax[0].set_title('% de valores sobre su media')
nh.loc[desde:].rolling(5).mean().plot(ax=ax[1], label='Máximos de 52 semanas', color='tab:green')
nl.loc[desde:].rolling(5).mean().plot(ax=ax[1], label='Mínimos de 52 semanas', color='tab:red'); ax[1].legend(); ax[1].set_title('% de valores (media de 5 sesiones)')
both.loc[desde:].plot(ax=ax[2], color='tab:purple', title='% bajo su MM50 y su MM200 a la vez'); ax[2].axhline(88, color='k', ls='--')
mcc.loc[desde:].plot(ax=ax[3], color='k', title='McClellan ajustado (19/39)'); ax[3].axhline(0, color='k', ls='--')
(spx / spx.rolling(252).max() - 1).loc[desde:].plot(ax=ax[4], color='k', ls='--', title='S&P 500: caída desde el máximo de 52 semanas')
plt.show()

In [ ]:
#@title Valores en máximos de 52 semanas frente a las caídas del S&P (semanal, alineado por fecha)
sem = px.resample('W-FRI').last()
mx = sem.rolling(52, min_periods=52).max()
n_max = share(sem >= mx * 0.99, mx.notna())
spx_w = spx.resample('W-FRI').last()
dd_w = spx_w / spx_w.rolling(52).max() - 1        # 52 semanas de verdad (antes, 364 sesiones)
both_w = pd.concat({'Caída del S&P': dd_w, '% en máximos': n_max}, axis=1).dropna()
corr = both_w.iloc[:, 0].rolling(13).corr(both_w.iloc[:, 1])   # 13 semanas: con 4 puntos era ruido
fig, ax = plt.subplots(3, 1, figsize=(16, 15), sharex=True)
both_w.iloc[:, 1].loc[desde:].plot(ax=ax[0], title='% de valores a menos de un 1% de su máximo de 52 semanas')
both_w.iloc[:, 0].loc[desde:].plot(ax=ax[1], color='salmon', title='S&P 500: caída desde el máximo de 52 semanas')
corr.loc[desde:].plot(ax=ax[2], color='tab:green', title='Correlación de 13 semanas'); ax[2].axhline(0, color='k', ls='--')
plt.show()
b = both_w.loc['2020':].reset_index(names='fecha')
plt.scatter(b['Caída del S&P'], b['% en máximos'], c=b['fecha'].astype('int64'), cmap='coolwarm', s=18)
plt.xlabel('Caída del S&P 500 desde su máximo de 52 semanas'); plt.ylabel('% de valores en máximos'); plt.title('Desde 2020 (azul = antes, rojo = después)')
plt.show()
print(both_w.corr().round(2))

## Valoración con Finviz: retirada
Finviz bloquea las descargas automáticas (403). Además el código, si fallaba una empresa, reutilizaba la página de la anterior,
ignoraba las capitalizaciones en billones («T»: Apple, Microsoft, Nvidia… desaparecían del cálculo), usaba el ticker FB y
`sp.multiples` en lugar de `self.multiples`. En su lugar, la rentabilidad por FCF tras SBC de FAAMG y de los semis frente al bono,
con los 10-K y 10-Q de EDGAR, está en la pestaña **Big Tech y semis** del monitor. La celda siguiente lee esos datos.

In [ ]:
#@title Datos de caja del monitor (EDGAR, ya calculados)
t = requests.get('https://boletix.github.io/recession-monitor/tech.json', timeout=30).json()
cos = pd.DataFrame({k: {**v['ltm'], 'último trimestre': v['last']} for k, v in t['cash']['companies'].items()}).T
cos['capex / caja op.'] = cos['capex'] / cos['ocf']
print('Últimos 12 meses, miles de millones de $ (EDGAR):')
print(cos[['último trimestre', 'rev', 'ocf', 'capex', 'capex / caja op.', 'fcf', 'sbc', 'fcf_sbc', 'bb', 'div', 'fin_rest']].round(2).to_string())
h = t['cash']['baskets']['hyper']['series']
pd.DataFrame({k: h[k] for k in ['ocf', 'capex', 'fcf_sbc']}, index=pd.to_datetime(h['dates'])).plot(
    title='Hiperescaladores: caja operativa, capex y FCF tras SBC (12 meses)'); plt.show()